In [58]:
from utils_extraction import tokenize, decode
from utils_extraction import is_auto_label_tag
from utils_extraction import extract_few_shot_examples_from_labels
from utils_extraction import select_few_shot 
from utils_extraction import process_labels
from utils_extraction import add_attributes_to_auto_labels, compare_html_allow_auto_labels
from models import GPTAssistant

In [59]:
# ---------- Define Hyperparameters ----------
model_name = "gpt-5.2"

n_few_shot = 20  # Number of few-shot examples to use

#### Define the text to process, and where to save it

In [79]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "2021QCCA1675"
anno = "llm"
version = "v1.2"
out_version = "v1.3"
html_path = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_2_500_100_30_gpt5.2\{filename}_llm_{version}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_2_500_100_30_gpt5.2"


# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "2019SCC65_annotated_EG_v1_corrected"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_30_gpt5.2\2021QCCA1675_llm_v1.2.html
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\2019SCC65_annotated_EG_v1_corrected.html


### Process The HTML Content

In [80]:
# ---------- Tokenize html content ----------
tokens = tokenize(html_content)

In [81]:
# ---------- Tokenize html content ----------
fs_tokens = tokenize(fs_html_content)

In [82]:
if out_version == "v1.1":
    sublabel_config = {
    "parent":["decision", "legislation", "secondary sources"], # only extract sublabels under these parents
    "already_labeled":[], # do not extract sublabels under these labels
    "new_labels":["title", "fragment"],
    "keep_attributes":["labelname"],
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.4, 0.4]
    
if out_version == "v1.2":
    sublabel_config = {
    "parent":["secondary sources"], # only extract sublabels under these parents
    "already_labeled":["title", "fragment"], # do not extract sublabels under these labels
    "new_labels":["source", "authors"],
    "keep_attributes":["labelname"], 
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.4, 0.4]

if out_version == "v1.3":
    sublabel_config = {
        "parent":["decision", "legislation"], # only extract sublabels under these parents
        "already_labeled":["title", "fragment", "source", "authors"], # do not extract sublabels under these labels
        "new_labels":["citation"],
        "keep_attributes":["labelname"], 
        "switch_type":True, # manual_label -> auto_label
        "use_simplified":True, # <auto_label labelname="title"> -> <title>
    }
    distribution=[0.8]

# Do not use remove_labels here, as we need the parent labels to identify sublabels : This could  create issues.

#### Get few shot

In [83]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples_from_labels(fs_tokens, 
                                              sublabel_config)


# Select examples with distributed method: 50% with "source" label, 50% random others
selected_few_shot_examples = select_few_shot(
    examples=few_shot_examples, 
    n=n_few_shot,
    method="distributed",
    list_of_labels=sublabel_config["new_labels"],
    distribution=distribution
)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")


   ✓ Selected 20 few-shot examples for processing.


In [84]:
selected_few_shot_examples

[('<decision><i><title>Canada (Attorney General) v. Almon Equipment Limited</title></i>, 2010 FCA 193, [2011] 4 F.C.R. 203</decision>',
  '<decision><i><title>Canada (Attorney General) v. Almon Equipment Limited</title></i>, <citation>2010 FCA 193</citation>, <citation>[2011] 4 F.C.R. 203</citation></decision>'),
 ('<decision><i><title>Mission Institution v. Khela</title></i>, 2014 SCC 24, [2014] 1 S.C.R. 502</decision>',
  '<decision><i><title>Mission Institution v. Khela</title></i>, <citation>2014 SCC 24</citation>, <citation>[2014] 1 S.C.R. 502</citation></decision>'),
 ('<decision><i><title>Wilson</title></i>, at <fragment>para. 25</fragment></decision>',
  '<decision><i><title>Wilson</title></i>, at <fragment>para. 25</fragment></decision>'),
 ('<decision>2019 SCC&nbsp;65</decision>',
  '<decision><citation>2019 SCC&nbsp;65</citation></decision>'),
 ('<decision> 2015 FC 960, [2016] 2 F.C.R. 39, 38 Imm. L.R. (4th) 110, [2015] F.C.J. No.&nbsp;981 (QL), 2015 CarswellNat 3740 (WL Can

#### Processing

In [85]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [86]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_sublabels_extraction_from_parent_cot.txt"

processed_label = process_labels(
    model=model,
    tokens=tokens,
    sublabel_config=sublabel_config,
    few_shot_examples=few_shot_examples,
    prompt_path=prompt_path,
    output_dir=output_dir,
    filename=filename,
)


   ✓ Found 92 parent mentions to process
   ✓ Built 185 token segments (92 to process)


Processing mentions:   0%|          | 0/185 [00:00<?, ?it/s]

Processing mentions: 100%|██████████| 185/185 [06:02<00:00,  1.96s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_30_gpt5.2\history_2021QCCA1675_sublabel.json
   ✓ Processed tokens saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_30_gpt5.2\processed_sublabels_2021QCCA1675.json

   ✓ Sublabel extraction completed:
      - Total mentions: 92
      - Successful: 92
      - Failed: 0


### Post Processing

In [87]:
# Useless verification to check if the tokens are the same after processing (except for the auto labels)
t1 = []
t2 = []
for token in processed_label:
    if not is_auto_label_tag(token) in [1, 2]:
        t1.append(token)


for token in tokens:
    if not is_auto_label_tag(token) in [1, 2]:
        t2.append(token)

assert t1 == t2, "The tokens are different after processing, which should not happen as we are only adding auto_label tags without changing the original tokens."


processed_html = decode(processed_label)

print(f"\HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = add_attributes_to_auto_labels(processed_html)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_{anno}_{out_version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")


<>:18: SyntaxWarning: invalid escape sequence '\H'
<>:18: SyntaxWarning: invalid escape sequence '\H'
C:\Users\zakga\AppData\Local\Temp\ipykernel_17988\1416758461.py:18: SyntaxWarning: invalid escape sequence '\H'
  print(f"\HTML length: {len(processed_html)}")


\HTML length: 129632
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_30_gpt5.2
